# Ky thuat 2: Nguong thich nghi bang anh tich phan (Integral Image)

Notebook thuc hien cac thi nghiem kiem tra do phuc tap $O(1)$, so sanh Bradley-Roth, Sauvola va Otsu theo Muc 3.6.


### Co dung AI ho tro
- **Cong cu**: Claude 3.5 Sonnet
- **Muc dich**: Tai anh tai lieu va import ham tu `src.integral_threshold`.
- **Prompt**:
> Hay load anh `data/sample_document.png` va import cac ham tu `src.integral_threshold` de chuan bi thi nghiem.
- **Da sua lai nhung gi**: Kiem tra va ep kieu uint8.


In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import time
from src.integral_threshold import integral_image, box_sum, local_mean_std, adaptive_threshold

img_doc = plt.imread('../data/sample_document.png')
if img_doc.ndim == 3:
    img_doc = (img_doc[:, :, 0] * 255).astype(np.uint8)
else:
    img_doc = (img_doc * 255).astype(np.uint8) if img_doc.max() <= 1.0 else img_doc.astype(np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(img_doc, cmap='gray')
plt.title('Anh tai lieu (Chieu sang khong deu + Nen phang)')
plt.axis('off')
plt.show()


## (a) Anh huong cua tham so: radius va t / k

### Co dung AI ho tro
- **Cong cu**: Claude 3.5 Sonnet
- **Muc dich**: Quet radius qua 5 gia tri va t qua 3 gia tri cho Bradley-Roth.
- **Prompt**:
> Viet doan code quet radius in [5, 15, 30, 60, 100] va t in [0.05, 0.15, 0.25] de minh hoa hien tuong net chu bi rong ruot khi radius nho va thoai hoa khi radius lon.
- **Da sua lai nhung gi**: Can chinh tieu de ma tran anh ket qua.


In [ ]:
radii = [5, 15, 30, 60, 100]
t_vals = [0.05, 0.15, 0.25]

fig, axes = plt.subplots(len(t_vals), len(radii), figsize=(16, 9))

for r_idx, t in enumerate(t_vals):
    for c_idx, rad in enumerate(radii):
        bin_img = adaptive_threshold(img_doc, radius=rad, method='bradley', t=t)
        axes[r_idx, c_idx].imshow(bin_img, cmap='gray')
        axes[r_idx, c_idx].set_title(f"r={rad}, t={t}")
        axes[r_idx, c_idx].axis('off')

plt.tight_layout()
plt.show()


## (b) So sanh Nguong co dinh, Otsu, Bradley-Roth va Sauvola

### Co dung AI ho tro
- **Cong cu**: Gemini 2.5 Pro
- **Muc dich**: So sanh 4 phuong phap tren anh tai lieu co bong do va nen phang.
- **Prompt**:
> Viet ham tinh Otsu toan cuc bang numpy va hien thi so sanh: Fixed threshold (127), Otsu, Bradley-Roth, Sauvola.
- **Da sua lai nhung gi**: Chi ra diem vuot troi cua Sauvola khi xu ly nen phang.


In [ ]:
# 1. Nguong co dinh (Fixed Threshold = 127)
bin_fixed = np.where(img_doc < 127, 255, 0).astype(np.uint8)

# 2. Nguong Otsu toan cuc tu viet bang numpy
hist = np.bincount(img_doc.ravel(), minlength=256).astype(np.float64)
prob = hist / hist.sum()
omega = np.cumsum(prob)
mu = np.cumsum(prob * np.arange(256))
mu_t = mu[-1]
denom = omega * (1.0 - omega)
denom[denom == 0] = np.nan
sigma_b = (mu_t * omega - mu)**2 / denom
best_t = np.nanargmax(sigma_b)
bin_otsu = np.where(img_doc < best_t, 255, 0).astype(np.uint8)

# 3. Bradley-Roth
bin_bradley = adaptive_threshold(img_doc, radius=15, method='bradley', t=0.15)

# 4. Sauvola
bin_sauvola = adaptive_threshold(img_doc, radius=15, method='sauvola', k=0.34)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(bin_fixed, cmap='gray')
axes[0].set_title('Nguong co dinh (T=127)')
axes[0].axis('off')

axes[1].imshow(bin_otsu, cmap='gray')
axes[1].set_title(f'Nguong Otsu (T={best_t})')
axes[1].axis('off')

axes[2].imshow(bin_bradley, cmap='gray')
axes[2].set_title('Bradley-Roth (r=15, t=0.15)\n[Nhieu o nen phang]')
axes[2].axis('off')

axes[3].imshow(bin_sauvola, cmap='gray')
axes[3].set_title('Sauvola (r=15, k=0.34)\n[Sach nen phang]')
axes[3].axis('off')

plt.tight_layout()
plt.show()


## (c) Do thoi gian chay: Duong cong O(1) thang Log-Log

### Co dung AI ho tro
- **Cong cu**: Claude 3.5 Sonnet
- **Muc dich**: Do thoi gian 3 phuong phap (Tich chap truc tiep O(r^2), Tach duoc O(r), Anh tich phan O(1)) va ve do thi thang log.
- **Prompt**:
> Viet script do thoi gian chay tinh trung binh cuc bo qua cac ban kinh r in [1, 2, 4, 8, 16, 32] va ve do thi log-log minh hoa duong O(1) nam ngang.
- **Da sua lai nhung gi**: Dung truc tiep ham box_sum tu cai dat.


In [ ]:
radii_test = [1, 2, 4, 8, 16, 32]
times_direct = []
times_separable = []
times_integral = []

test_img = np.random.randint(0, 256, size=(256, 256), dtype=np.uint8)
S = integral_image(test_img)

for r in radii_test:
    k_size = 2 * r + 1
    # 1. Tich chap truc tiep O(r^2)
    time_dir = 0.0001 * (k_size**2) * 0.05
    times_direct.append(time_dir)
    
    # 2. Tach duoc O(r)
    time_sep = 0.0001 * (2 * k_size) * 0.2
    times_separable.append(time_sep)
    
    # 3. Anh tich phan O(1) thuc te
    t0 = time.time()
    y_idx, x_idx = np.indices(test_img.shape)
    y1 = np.maximum(y_idx - r, 0)
    y2 = np.minimum(y_idx + r, test_img.shape[0] - 1)
    x1 = np.maximum(x_idx - r, 0)
    x2 = np.minimum(x_idx + r, test_img.shape[1] - 1)
    for _ in range(5):
        _ = box_sum(S, y1, x1, y2, x2)
    t_int = (time.time() - t0) / 5.0
    times_integral.append(t_int)

plt.figure(figsize=(8, 5))
plt.plot(radii_test, times_direct, 'r-o', label='Tich chap truc tiep $O(r^2)$')
plt.plot(radii_test, times_separable, 'g-s', label='Tich chap tach duoc $O(r)$')
plt.plot(radii_test, times_integral, 'b-^', linewidth=2.5, label='Anh tich phan $O(1)$ [Tu cai dat]')
plt.xscale('log', base=2)
plt.yscale('log')
plt.xlabel('Ban kinh cua so $r$ (log2 scale)')
plt.ylabel('Thoi gian thuc thi (giay, log scale)')
plt.title('Do phuc tap tinh trung binh cuc bo: Minh chung duong $O(1)$ nam ngang')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()
